# Introducción a Regresión Lineal con Scikit-Learn

**Elaborado por:** David Palacio J.  
**Correo:** davidpalacioj@gmail.com

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dpalacioj/DataAI-Fundamentos-Aplicaciones/blob/main/Analitica/ciencia_datos_con_python/notebooks_teoria/09_1_intro_regresion_lineal.ipynb)

---

## 🎯 ¿Qué es la Regresión Lineal?

La **regresión lineal** es uno de los algoritmos más fundamentales en Machine Learning. Su objetivo es **predecir un valor numérico** basándose en una o más características de entrada.

### 📈 **Conceptos Clave:**

**🔢 Variable Dependiente (Y)**: Lo que queremos predecir
- Ejemplos: precio de una casa, salario, temperatura

**📊 Variables Independientes (X)**: Las características que usamos para predecir
- Ejemplos: metros cuadrados, años de experiencia, humedad

**📐 La Ecuación Básica**:
```
Y = β₀ + β₁X₁ + β₂X₂ + ... + βₙXₙ + ε
```

Donde:
- **β₀**: intercepto (valor cuando todas las X son cero)
- **β₁, β₂, ..., βₙ**: coeficientes (pendientes)
- **ε**: error aleatorio

## 🛠️ Instalación y Librerías

Vamos a usar las librerías más importantes para Machine Learning en Python:

In [ ]:
# Instalar librerías necesarias (solo en Colab)
# !pip install scikit-learn pandas numpy matplotlib seaborn plotly --quiet

# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Scikit-Learn para Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# Configuración
import warnings
warnings.filterwarnings('ignore')
plt.style.use('default')
np.random.seed(42)

print("✅ Librerías importadas exitosamente")

## 🏠 Dataset: Precios de Casas en Boston

Vamos a usar el famoso dataset de precios de casas en Boston. Aunque este dataset tiene algunas limitaciones éticas (que discutiremos), es excelente para aprender regresión lineal.

### 📊 **Características del Dataset:**
- **506 observaciones** de diferentes vecindarios
- **13 características** como crimen, edad de casas, acceso a autopistas
- **Variable objetivo**: precio medio de casas (en miles de USD, año 1970)

### 📝 **Nota Importante:**
Este dataset se usa únicamente con fines educativos para aprender técnicas de ML.

In [ ]:
# Cargar el dataset desde GitHub (alternativa al dataset de sklearn)
url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
df = pd.read_csv(url)

print(f"📊 Forma del dataset: {df.shape}")
print(f"📋 Columnas: {df.shape[1]}")
print(f"🏘️ Observaciones: {df.shape[0]}")

print("\n📈 Información general del dataset:")
print(df.info())

print("\n📝 Primeras 5 filas:")
df.head()

### 📖 **Descripción de Variables**

| Variable | Descripción |
|----------|-------------|
| **crim** | Tasa de crimen per cápita |
| **zn** | Proporción de zonas residenciales |
| **indus** | Proporción de negocios no retail |
| **chas** | Variable dummy río Charles (1 si toca el río) |
| **nox** | Concentración de óxidos nítricos |
| **rm** | Número promedio de habitaciones por vivienda |
| **age** | Proporción de unidades construidas antes de 1940 |
| **dis** | Distancia a centros de empleo |
| **rad** | Índice de accesibilidad a autopistas |
| **tax** | Tasa de impuesto a la propiedad |
| **ptratio** | Ratio alumno-profesor |
| **lstat** | % de población de bajo estatus socioeconómico |
| **medv** | 🎯 **OBJETIVO**: Valor medio de casas (miles USD) |

In [ ]:
# Estadísticas descriptivas
print("📊 Estadísticas descriptivas:")
df.describe().round(2)

## 🔍 Análisis Exploratorio de Datos (EDA)

Antes de construir nuestro modelo, necesitamos entender los datos:

In [ ]:
# Verificar valores faltantes
print("🔍 Valores faltantes por columna:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "✅ No hay valores faltantes")

# Distribución de la variable objetivo
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Distribución de Precios', 'Boxplot de Precios']
)

# Histograma
fig.add_trace(
    go.Histogram(x=df['medv'], name='Precios', nbinsx=30),
    row=1, col=1
)

# Boxplot
fig.add_trace(
    go.Box(y=df['medv'], name='Precios'),
    row=1, col=2
)

fig.update_layout(
    title_text="Análisis de la Variable Objetivo: Precio de Casas",
    showlegend=False,
    height=400
)
fig.show()

print(f"💰 Precio promedio: ${df['medv'].mean():.1f}k")
print(f"📈 Precio máximo: ${df['medv'].max():.1f}k")
print(f"📉 Precio mínimo: ${df['medv'].min():.1f}k")
print(f"📊 Desviación estándar: ${df['medv'].std():.1f}k")

In [ ]:
# Matriz de correlación
correlation_matrix = df.corr()

# Crear heatmap interactivo
fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=correlation_matrix.round(2).values,
    texttemplate="%{text}",
    textfont={"size":10}
))

fig.update_layout(
    title="Matriz de Correlación - Dataset Boston Housing",
    height=600,
    width=800
)
fig.show()

# Variables más correlacionadas con el precio
price_corr = correlation_matrix['medv'].abs().sort_values(ascending=False)[1:]  # Excluir la correlación consigo misma
print("🎯 Variables más correlacionadas con el precio:")
for var, corr in price_corr.head(5).items():
    print(f"  {var}: {corr:.3f}")

### 🔎 **Interpretación de Correlaciones:**

**Correlaciones Positivas Fuertes** (🔵 azul):
- **rm** (habitaciones): Más habitaciones → precio más alto ✅

**Correlaciones Negativas Fuertes** (🔴 rojo):
- **lstat** (bajo estatus): Más pobreza → precio más bajo ✅
- **crim** (crimen): Más crimen → precio más bajo ✅

In [ ]:
# Scatter plots de las variables más importantes
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Habitaciones vs Precio', 'Crimen vs Precio', 
                   'Estatus Socioeconómico vs Precio', 'Edad de Casas vs Precio']
)

variables_importantes = ['rm', 'crim', 'lstat', 'age']
titles = ['Habitaciones', 'Crimen', 'Bajo Estatus %', 'Edad Casas']

for i, (var, title) in enumerate(zip(variables_importantes, titles)):
    row = i // 2 + 1
    col = i % 2 + 1
    
    fig.add_trace(
        go.Scatter(
            x=df[var], 
            y=df['medv'],
            mode='markers',
            name=f'{title}',
            opacity=0.6
        ),
        row=row, col=col
    )

fig.update_layout(
    title_text="Relación entre Variables Clave y Precio de Casas",
    height=600,
    showlegend=False
)
fig.show()

## 🤖 Construyendo el Modelo de Regresión Lineal

### 📊 **Paso 1: Preparar los Datos**

Dividiremos los datos en:
- **X**: Variables independientes (características)
- **y**: Variable dependiente (precio)

In [ ]:
# Separar características (X) y variable objetivo (y)
X = df.drop('medv', axis=1)  # Todas las columnas excepto 'medv'
y = df['medv']               # Solo la columna 'medv'

print(f"📊 Forma de X (características): {X.shape}")
print(f"🎯 Forma de y (objetivo): {y.shape}")
print(f"\n📋 Características utilizadas: {list(X.columns)}")

### 📊 **Paso 2: Dividir en Entrenamiento y Prueba**

Es fundamental dividir nuestros datos para **evaluar** qué tan bien funciona el modelo con datos que nunca ha visto:

In [ ]:
# Dividir en conjunto de entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,        # 20% para pruebas
    random_state=42       # Para reproducibilidad
)

print("📊 División de datos completada:")
print(f"  🏋️ Entrenamiento: {X_train.shape[0]} observaciones")
print(f"  🧪 Prueba: {X_test.shape[0]} observaciones")
print(f"  📊 Proporción: {X_train.shape[0]/(X_train.shape[0]+X_test.shape[0]):.1%} / {X_test.shape[0]/(X_train.shape[0]+X_test.shape[0]):.1%}")

### 🤖 **Paso 3: Entrenar el Modelo**

Aquí es donde la "magia" sucede. El algoritmo encontrará los mejores coeficientes para hacer predicciones:

In [ ]:
# Crear y entrenar el modelo
modelo = LinearRegression()

# Entrenar el modelo con datos de entrenamiento
print("🚀 Entrenando modelo de regresión lineal...")
modelo.fit(X_train, y_train)
print("✅ Modelo entrenado exitosamente!")

# Información sobre el modelo entrenado
print(f"\n📊 Intercepto (β₀): ${modelo.intercept_:.2f}k")
print(f"📈 Número de coeficientes: {len(modelo.coef_)}")

### 📊 **Interpretando los Coeficientes**

Los coeficientes nos dicen **cuánto cambia el precio** cuando cada variable aumenta en 1 unidad:

In [ ]:
# Crear DataFrame con coeficientes para mejor visualización
coeficientes = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente': modelo.coef_,
    'Impacto_Abs': np.abs(modelo.coef_)
}).sort_values('Impacto_Abs', ascending=False)

print("📊 Coeficientes del modelo (ordenados por impacto):")
for _, row in coeficientes.head(8).iterrows():
    variable = row['Variable']
    coef = row['Coeficiente']
    signo = "📈" if coef > 0 else "📉"
    print(f"  {signo} {variable}: {coef:+6.2f}k por unidad")

# Visualizar coeficientes
fig = go.Figure()
colors = ['green' if coef > 0 else 'red' for coef in coeficientes['Coeficiente']]

fig.add_trace(go.Bar(
    x=coeficientes['Variable'],
    y=coeficientes['Coeficiente'],
    marker_color=colors,
    text=coeficientes['Coeficiente'].round(2),
    textposition='outside'
))

fig.update_layout(
    title="Coeficientes del Modelo de Regresión Lineal",
    xaxis_title="Variables",
    yaxis_title="Coeficiente (cambio en precio por unidad)",
    height=500
)
fig.show()

### 💡 **Interpretación de Coeficientes Clave:**

**📈 Coeficientes Positivos (aumentan el precio):**
- **rm**: +1 habitación → +$X,XXXk en precio
- **chas**: Estar cerca del río → precio más alto

**📉 Coeficientes Negativos (disminuyen el precio):**
- **lstat**: +1% de pobreza → -$XXXk en precio  
- **crim**: +1 unidad de crimen → precio más bajo

## 🎯 Haciendo Predicciones y Evaluando el Modelo

### 🔮 **Paso 4: Hacer Predicciones**

In [ ]:
# Hacer predicciones en ambos conjuntos
y_train_pred = modelo.predict(X_train)
y_test_pred = modelo.predict(X_test)

print("🔮 Predicciones realizadas:")
print(f"  📊 Entrenamiento: {len(y_train_pred)} predicciones")
print(f"  🧪 Prueba: {len(y_test_pred)} predicciones")

# Mostrar algunas predicciones vs valores reales
comparacion = pd.DataFrame({
    'Precio_Real': y_test.values[:10],
    'Precio_Predicho': y_test_pred[:10],
    'Diferencia': y_test.values[:10] - y_test_pred[:10]
})

print("\n📊 Primeras 10 predicciones vs valores reales:")
print(comparacion.round(2))

### 📏 **Paso 5: Métricas de Evaluación**

Necesitamos medir qué tan bien funciona nuestro modelo:

In [ ]:
# Calcular métricas de evaluación
def calcular_metricas(y_real, y_pred, nombre_conjunto):
    mse = mean_squared_error(y_real, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_real, y_pred)
    r2 = r2_score(y_real, y_pred)
    
    print(f"\n📊 Métricas para conjunto de {nombre_conjunto}:")
    print(f"  📏 MAE (Error Absoluto Medio): ${mae:.2f}k")
    print(f"  📐 RMSE (Raíz Error Cuadrático Medio): ${rmse:.2f}k")
    print(f"  🎯 R² (Coeficiente de Determinación): {r2:.3f}")
    print(f"  📈 Varianza Explicada: {r2*100:.1f}%")
    
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

# Evaluar en ambos conjuntos
metricas_train = calcular_metricas(y_train, y_train_pred, "entrenamiento")
metricas_test = calcular_metricas(y_test, y_test_pred, "prueba")

# Crear comparación visual
metricas_df = pd.DataFrame({
    'Entrenamiento': [metricas_train['MAE'], metricas_train['RMSE'], metricas_train['R2']],
    'Prueba': [metricas_test['MAE'], metricas_test['RMSE'], metricas_test['R2']]
}, index=['MAE ($k)', 'RMSE ($k)', 'R²'])

print("\n📋 Comparación de métricas:")
print(metricas_df.round(3))

### 📊 **Interpretación de Métricas:**

**📏 MAE (Error Absoluto Medio)**:
- Promedio de errores en términos absolutos
- Más fácil de interpretar: "En promedio, nos equivocamos por $X,XXX"

**📐 RMSE (Raíz del Error Cuadrático Medio)**:
- Penaliza más los errores grandes
- Misma unidad que la variable objetivo

**🎯 R² (Coeficiente de Determinación)**:
- Va de 0 a 1 (mientras más cerca de 1, mejor)
- Indica qué % de la variabilidad explica el modelo

In [ ]:
# Gráfico de predicciones vs valores reales
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Entrenamiento', 'Prueba']
)

# Scatter plot para entrenamiento
fig.add_trace(
    go.Scatter(
        x=y_train, 
        y=y_train_pred,
        mode='markers',
        name='Entrenamiento',
        opacity=0.6
    ),
    row=1, col=1
)

# Scatter plot para prueba
fig.add_trace(
    go.Scatter(
        x=y_test, 
        y=y_test_pred,
        mode='markers',
        name='Prueba',
        opacity=0.6
    ),
    row=1, col=2
)

# Línea de predicción perfecta (y = x)
min_val = min(y.min(), min(y_train_pred.min(), y_test_pred.min()))
max_val = max(y.max(), max(y_train_pred.max(), y_test_pred.max()))

for col in [1, 2]:
    fig.add_trace(
        go.Scatter(
            x=[min_val, max_val],
            y=[min_val, max_val],
            mode='lines',
            name='Predicción Perfecta',
            line=dict(color='red', dash='dash'),
            showlegend=(col==1)
        ),
        row=1, col=col
    )

fig.update_layout(
    title_text="Predicciones vs Valores Reales",
    height=500
)

# Actualizar ejes
fig.update_xaxes(title_text="Precio Real ($k)")
fig.update_yaxes(title_text="Precio Predicho ($k)")

fig.show()

print("📊 Interpretación del gráfico:")
print("  🔴 Línea roja: predicción perfecta (donde predicho = real)")
print("  📊 Puntos cerca de la línea: buenas predicciones")
print("  ⚠️ Puntos lejos de la línea: predicciones con mayor error")

## 🔍 Análisis de Residuales

Los **residuales** son la diferencia entre los valores reales y las predicciones. Analizarlos nos ayuda a entender si nuestro modelo tiene problemas:

In [ ]:
# Calcular residuales
residuales_train = y_train - y_train_pred
residuales_test = y_test - y_test_pred

# Gráfico de residuales
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Residuales vs Predicciones (Entrenamiento)', 
                   'Residuales vs Predicciones (Prueba)',
                   'Distribución Residuales (Entrenamiento)',
                   'Distribución Residuales (Prueba)']
)

# Residuales vs predicciones
fig.add_trace(
    go.Scatter(x=y_train_pred, y=residuales_train, mode='markers', 
               name='Train', opacity=0.6),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=y_test_pred, y=residuales_test, mode='markers', 
               name='Test', opacity=0.6),
    row=1, col=2
)

# Histogramas de residuales
fig.add_trace(
    go.Histogram(x=residuales_train, name='Train Hist', nbinsx=20),
    row=2, col=1
)

fig.add_trace(
    go.Histogram(x=residuales_test, name='Test Hist', nbinsx=20),
    row=2, col=2
)

# Línea horizontal en y=0 para los scatter plots
for col in [1, 2]:
    fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=col)

fig.update_layout(
    title_text="Análisis de Residuales",
    height=600,
    showlegend=False
)

fig.show()

print("🔍 ¿Qué buscamos en los residuales?")
print("  ✅ Distribución alrededor de cero (línea roja)")
print("  ✅ No patrones obvios en el scatter plot")
print("  ✅ Distribución aproximadamente normal en histograma")
print(f"\n📊 Estadísticas de residuales:")
print(f"  📏 Media residuales prueba: {residuales_test.mean():.3f}")
print(f"  📐 Desv. std residuales prueba: {residuales_test.std():.3f}")

## 🏠 Ejemplo Práctico: Predecir el Precio de una Casa

Vamos a usar nuestro modelo para predecir el precio de una casa con características específicas:

In [ ]:
# Definir características de una casa ejemplo
casa_ejemplo = {
    'crim': 0.1,        # Baja tasa de crimen
    'zn': 20.0,         # 20% zona residencial
    'indus': 5.0,       # 5% industria
    'chas': 1,          # Cerca del río
    'nox': 0.4,         # Baja contaminación
    'rm': 7.0,          # 7 habitaciones (bueno!)
    'age': 30.0,        # 30% casas viejas
    'dis': 4.0,         # Distancia media a empleos
    'rad': 3,           # Acceso medio a autopistas
    'tax': 250,         # Impuestos medios
    'ptratio': 16.0,    # Ratio estudiante-profesor
    'lstat': 5.0        # 5% pobreza (bajo)
}

# Convertir a DataFrame (mismo formato que datos de entrenamiento)
casa_df = pd.DataFrame([casa_ejemplo])

# Hacer predicción
precio_predicho = modelo.predict(casa_df)[0]

print("🏠 Características de la casa ejemplo:")
for caracteristica, valor in casa_ejemplo.items():
    print(f"  📊 {caracteristica}: {valor}")

print(f"\n💰 Precio predicho: ${precio_predicho:.1f}k")
print(f"💵 En USD actuales (~2024): ${precio_predicho * 5:.0f}k")
print("\n📝 Nota: El dataset original es de 1970, por eso multiplicamos por ~5 para aproximar valores actuales")

# Comparar con estadísticas del dataset
print(f"\n📊 Comparación con el dataset:")
print(f"  📈 Precio promedio dataset: ${df['medv'].mean():.1f}k")
print(f"  📊 Percentil de nuestra predicción: {(df['medv'] < precio_predicho).mean()*100:.0f}%")
print(f"  💡 Nuestra casa está en el percentil {(df['medv'] < precio_predicho).mean()*100:.0f}, ¡bastante buena!")

## 🎯 ¿Cómo Mejorar el Modelo?

### 🔧 **Técnicas de Mejora:**

1. **📊 Ingeniería de Características**:
   - Crear nuevas variables combinando existentes
   - Transformaciones (log, sqrt, polinomiales)
   - Normalización/Estandarización

2. **🤖 Algoritmos Más Complejos**:
   - Regresión Ridge/Lasso (regularización)
   - Random Forest
   - Gradient Boosting

3. **🔍 Mejor Selección de Variables**:
   - Eliminar variables poco importantes
   - Análisis de multicolinealidad
   - Validación cruzada

In [ ]:
# Ejemplo simple: Regresión con solo las 5 variables más importantes
from sklearn.feature_selection import SelectKBest, f_regression

# Seleccionar las 5 mejores características
selector = SelectKBest(score_func=f_regression, k=5)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

# Entrenar modelo con características seleccionadas
modelo_simple = LinearRegression()
modelo_simple.fit(X_train_selected, y_train)

# Predicciones
y_test_pred_simple = modelo_simple.predict(X_test_selected)

# Evaluar
r2_simple = r2_score(y_test, y_test_pred_simple)
rmse_simple = np.sqrt(mean_squared_error(y_test, y_test_pred_simple))

# Variables seleccionadas
variables_seleccionadas = X.columns[selector.get_support()]

print("🎯 Modelo Simplificado (5 mejores variables):")
print(f"  📊 Variables: {list(variables_seleccionadas)}")
print(f"  🎯 R²: {r2_simple:.3f}")
print(f"  📐 RMSE: ${rmse_simple:.2f}k")

print(f"\n📊 Comparación:")
print(f"  🤖 Modelo completo R²: {metricas_test['R2']:.3f}")
print(f"  ⚡ Modelo simple R²: {r2_simple:.3f}")
print(f"  📈 Diferencia: {(metricas_test['R2'] - r2_simple)*100:+.1f} puntos porcentuales")

if r2_simple > 0.9 * metricas_test['R2']:
    print("  ✅ ¡El modelo simple funciona casi igual de bien!")
else:
    print("  ⚠️ El modelo completo funciona significativamente mejor")

## 🎓 Resumen y Conceptos Clave

### 📚 **Lo que Aprendimos:**

1. **🎯 Regresión Lineal**: Predice valores numéricos usando relaciones lineales
2. **📊 Preparación de Datos**: Dividir en entrenamiento/prueba es crucial
3. **🤖 Scikit-Learn**: Librería poderosa y fácil de usar para ML
4. **📏 Métricas**: MAE, RMSE, R² nos dicen qué tan bien funciona el modelo
5. **🔍 Interpretación**: Los coeficientes nos explican qué variables son importantes

### 💡 **Cuándo Usar Regresión Lineal:**

**✅ Buena opción cuando:**
- Relación lineal entre variables
- Necesitas interpretabilidad
- Dataset pequeño/mediano
- Variables numéricas continuas

**❌ Considera otras opciones cuando:**
- Relaciones no lineales complejas
- Muchas variables categóricas
- Dataset muy grande
- Necesitas máxima precisión

### 🚀 **Próximos Pasos Recomendados:**

1. **🔧 Practicar con otros datasets**: Salarios, ventas, datos financieros
2. **📊 Ingeniería de características**: Crear nuevas variables útiles
3. **🤖 Probar otros algoritmos**: Random Forest, Gradient Boosting
4. **📈 Validación cruzada**: Técnica más robusta para evaluar modelos
5. **🔍 Regularización**: Ridge y Lasso para evitar overfitting

### 📖 **Recursos Adicionales:**

- [Documentación Scikit-Learn](https://scikit-learn.org/stable/modules/linear_model.html)
- [Kaggle Learn - Intro to Machine Learning](https://www.kaggle.com/learn/intro-to-machine-learning)
- [Estadística para Data Science](https://www.khanacademy.org/math/statistics-probability)

---

**¡Ahora tienes las bases sólidas para hacer predicciones con Machine Learning! 🎯**